# FinSmart — FinBot dan Sistem Rekomendasi
**Muhammad Syaiful**

---

| No | Bagian | Section |
|----|--------|---------|
| 1 | Install dan Import Library | Section 1 |
| 2 | Load Dataset | Section 2 |
| 3 | FinBot — Chatbot Keuangan | Section 3-5 |
| 4 | Sistem Rekomendasi Kesiapan Investasi | Section 6-7 |
| 5 | Dokumentasi Arsitektur Pipeline AI | Section 8 |
| 6 | Ringkasan | Section 9 |

Notebook ini dijalankan setelah `FinSmart_AI_Engineer_UPDATED_v2.ipynb` selesai karena membutuhkan file `encoders.pkl`, `scaler.pkl`, `xgb_model.pkl`, dan `financial_type_classifier.pkl`.


## 1. Install dan Import Library

In [1]:
!pip install transformers torch scikit-learn fastapi uvicorn -q

print("Instalasi selesai")

Instalasi selesai


In [2]:
import numpy as np
import pandas as pd
import json
import os
import pickle
import joblib
from transformers import pipeline

print("Semua library berhasil diimport")

Semua library berhasil diimport


## 2. Load Dataset

In [3]:
df = pd.read_csv("data_TransactionCategorizationuntukmodel.csv")

print(f"Shape dataset : {df.shape}")
print(f"Kolom         : {list(df.columns)}")
df.head(3)

Shape dataset : (8000, 8)
Kolom         : ['Amount', 'Payment_Method', 'Week_Day', 'Month', 'Time_Of_Day', 'MerchantName', 'Day', 'Category']


,Amount,Payment_Method,Week_Day,Month,Time_Of_Day,MerchantName,Day,Category
0,2.773539,1,6,4,2,81,1,Electronics
1,-0.420496,3,6,4,1,18,1,Transport
2,0.254552,2,6,4,0,28,1,Healthcare


## 3. FinBot — Definisi Intent dan Database Respons

FinBot menggunakan pendekatan Zero-Shot Classification dari Hugging Face untuk mendeteksi intent dari pertanyaan pengguna, kemudian memetakannya ke respons yang telah didefinisikan.

Model yang digunakan: `facebook/bart-large-mnli`


In [4]:
import warnings
warnings.filterwarnings("ignore")

print("Loading model Hugging Face, harap tunggu...")

finbot_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Model FinBot berhasil diload")

Loading model Hugging Face, harap tunggu...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model FinBot berhasil diload


In [5]:
INTENT_LABELS = [
    "tips menabung",
    "cara mengatur anggaran",
    "kategori pengeluaran",
    "investasi untuk pemula",
    "cara mengurangi pengeluaran",
    "target tabungan",
    "rekomendasi keuangan",
    "pengertian literasi keuangan",
]

RESPONSE_DATABASE = {
    "tips menabung": (
        "Tips Menabung yang Efektif:\n"
        "1. Terapkan metode 50/30/20 — 50% kebutuhan, 30% keinginan, 20% tabungan.\n"
        "2. Simpan di awal bulan sebelum digunakan (pay yourself first).\n"
        "3. Buat rekening tabungan terpisah dari rekening harian.\n"
        "4. Mulai dari nominal kecil tapi konsisten — Rp 50.000/hari = Rp 1,5 juta/bulan.\n"
        "5. Gunakan fitur Target Tabungan di FinSmart untuk pantau progres kamu."
    ),
    "cara mengatur anggaran": (
        "Cara Mengatur Anggaran Bulanan:\n"
        "1. Catat semua pemasukan bulanan terlebih dahulu.\n"
        "2. Bagi pengeluaran ke kategori: Makanan, Transport, Tagihan, Hiburan, dll.\n"
        "3. Tentukan batas maksimal untuk setiap kategori.\n"
        "4. Pantau pengeluaran harian agar tidak melebihi batas.\n"
        "5. FinSmart dapat mengingatkan kamu saat mendekati batas anggaran."
    ),
    "kategori pengeluaran": (
        "Kategori Pengeluaran di FinSmart:\n"
        "FinSmart mengklasifikasikan transaksi ke 10 kategori:\n"
        "1. Food | 2. Grocery | 3. Transport | 4. Entertainment | 5. Healthcare\n"
        "6. Electronics | 7. Online Shopping | 8. Travel | 9. Clothing | 10. Bills\n\n"
        "Kategori ditentukan otomatis oleh AI saat kamu input transaksi."
    ),
    "investasi untuk pemula": (
        "Panduan Investasi untuk Pemula:\n"
        "Sebelum investasi, pastikan sudah:\n"
        "- Punya dana darurat (3-6x pengeluaran bulanan)\n"
        "- Tidak punya hutang konsumtif\n"
        "- Tabungan rutin sudah berjalan\n\n"
        "Pilihan investasi untuk pemula:\n"
        "- Reksa Dana Pasar Uang — risiko rendah, cocok untuk pemula\n"
        "- Deposito — aman, bunga pasti\n"
        "- Obligasi Negara (ORI/SBR) — dijamin pemerintah\n\n"
        "Cek fitur Kesiapan Investasi di FinSmart untuk analisis profil keuanganmu."
    ),
    "cara mengurangi pengeluaran": (
        "Tips Mengurangi Pengeluaran:\n"
        "1. Identifikasi kategori pengeluaran terbesar dari dashboard FinSmart.\n"
        "2. Bedakan kebutuhan (harus) vs keinginan (bisa ditunda).\n"
        "3. Kurangi frekuensi makan di luar — masak sendiri bisa hemat 60%.\n"
        "4. Manfaatkan promo dan cashback saat belanja online.\n"
        "5. Batalkan langganan yang jarang dipakai."
    ),
    "target tabungan": (
        "Cara Menetapkan Target Tabungan:\n"
        "1. Tentukan tujuan spesifik: liburan, DP rumah, dana darurat, dll.\n"
        "2. Hitung nominal yang dibutuhkan dan tenggat waktunya.\n"
        "3. Bagi dengan jumlah bulan untuk mendapatkan target bulanan.\n"
        "4. Set target di fitur Sistem Anggaran FinSmart.\n"
        "5. FinSmart akan memberi notifikasi jika kamu on-track."
    ),
    "rekomendasi keuangan": (
        "Rekomendasi Keuangan Personal:\n"
        "Berdasarkan data keuanganmu, FinSmart memberikan rekomendasi:\n"
        "- Analisis pola pengeluaran bulanan\n"
        "- Kategori mana yang paling banyak menguras anggaran\n"
        "- Rekomendasi batas anggaran per kategori\n\n"
        "Masuk ke menu Analisis dan Rekomendasi AI untuk melihat insight personalmu."
    ),
    "pengertian literasi keuangan": (
        "Apa itu Literasi Keuangan?\n"
        "Literasi keuangan adalah kemampuan memahami dan menggunakan\n"
        "berbagai konsep keuangan secara efektif, meliputi:\n"
        "- Mengelola pendapatan dan pengeluaran\n"
        "- Merencanakan tabungan dan investasi\n"
        "- Memahami produk keuangan (kredit, asuransi, investasi)\n"
        "- Membuat keputusan keuangan yang bijak\n\n"
        "Data OJK 2024: literasi keuangan generasi muda Indonesia masih di bawah 50%."
    ),
}

RESPONSE_DEFAULT = (
    "Maaf, saya belum bisa menjawab pertanyaan tersebut.\n"
    "Kamu bisa tanya tentang: tips menabung, mengatur anggaran, "
    "kategori pengeluaran, investasi pemula, atau target tabungan."
)

print(f"Database FinBot siap: {len(RESPONSE_DATABASE)} intent terdaftar")

Database FinBot siap: 8 intent terdaftar


## 4. Kelas FinBot

Kelas `FinBot` mengelola alur deteksi intent dan pengembalian respons. Setiap percakapan disimpan dalam `chat_history` untuk keperluan logging.


In [6]:
class FinBot:
    """
    Chatbot keuangan FinSmart berbasis Hugging Face Zero-Shot Classification.

    Alur kerja:
    1. Terima pertanyaan pengguna
    2. Deteksi intent menggunakan Zero-Shot Classification
    3. Kembalikan respons sesuai intent yang terdeteksi
    """

    def __init__(self, classifier, intent_labels, response_db, threshold=0.10):
        self.classifier    = classifier
        self.intent_labels = intent_labels
        self.response_db   = response_db
        self.threshold     = threshold
        self.chat_history  = []

    def detect_intent(self, pertanyaan: str) -> tuple:
        result     = self.classifier(
            pertanyaan,
            candidate_labels=self.intent_labels,
            hypothesis_template="Pertanyaan ini tentang {}."
        )
        top_intent = result["labels"][0]
        top_score  = result["scores"][0]
        return top_intent, top_score

    def chat(self, pertanyaan: str) -> dict:
        pertanyaan = pertanyaan.strip()
        if not pertanyaan:
            return {"error": "Pertanyaan tidak boleh kosong."}

        intent, confidence = self.detect_intent(pertanyaan)

        if confidence >= self.threshold:
            respons = self.response_db.get(intent, RESPONSE_DEFAULT)
        else:
            respons = RESPONSE_DEFAULT
            intent  = "tidak_terdeteksi"

        entry = {
            "pertanyaan" : pertanyaan,
            "intent"     : intent,
            "confidence" : round(confidence * 100, 2),
            "respons"    : respons,
        }
        self.chat_history.append(entry)
        return entry

    def tampilkan_respons(self, result: dict):
        print("=" * 60)
        print(f"Pertanyaan : {result['pertanyaan']}")
        print(f"Intent     : {result['intent']} ({result['confidence']}%)")
        print("-" * 60)
        print(f"FinBot     :\n{result['respons']}")
        print("=" * 60)


finbot = FinBot(
    classifier    = finbot_classifier,
    intent_labels = INTENT_LABELS,
    response_db   = RESPONSE_DATABASE,
    threshold     = 0.10
)

print("FinBot berhasil diinisialisasi")

FinBot berhasil diinisialisasi


## 5. Uji FinBot

In [7]:
pertanyaan_uji = [
    "Gimana cara biar bisa nabung lebih banyak tiap bulan?",
    "Aku mau mulai investasi tapi masih pemula, harus mulai dari mana?",
    "Pengeluaran makanku terlalu besar, gimana cara nguranginnya?",
    "Apa itu kategori pengeluaran di FinSmart?",
    "Bantu aku bikin target tabungan untuk liburan",
]

for pertanyaan in pertanyaan_uji:
    result = finbot.chat(pertanyaan)
    finbot.tampilkan_respons(result)
    print()

Pertanyaan : Gimana cara biar bisa nabung lebih banyak tiap bulan?
Intent     : rekomendasi keuangan (15.88%)
------------------------------------------------------------
FinBot     :
Rekomendasi Keuangan Personal:
Berdasarkan data keuanganmu, FinSmart memberikan rekomendasi:
- Analisis pola pengeluaran bulanan
- Kategori mana yang paling banyak menguras anggaran
- Rekomendasi batas anggaran per kategori

Masuk ke menu Analisis dan Rekomendasi AI untuk melihat insight personalmu.

Pertanyaan : Aku mau mulai investasi tapi masih pemula, harus mulai dari mana?
Intent     : investasi untuk pemula (23.45%)
------------------------------------------------------------
FinBot     :
Panduan Investasi untuk Pemula:
Sebelum investasi, pastikan sudah:
- Punya dana darurat (3-6x pengeluaran bulanan)
- Tidak punya hutang konsumtif
- Tabungan rutin sudah berjalan

Pilihan investasi untuk pemula:
- Reksa Dana Pasar Uang — risiko rendah, cocok untuk pemula
- Deposito — aman, bunga pasti
- Obligasi Neg

In [8]:
with open("finbot_chat_history.json", "w", encoding="utf-8") as f:
    json.dump(finbot.chat_history, f, ensure_ascii=False, indent=2)

print(f"Chat history disimpan   : finbot_chat_history.json")
print(f"Total percakapan        : {len(finbot.chat_history)}")

Chat history disimpan   : finbot_chat_history.json
Total percakapan        : 5


## 6. Sistem Rekomendasi Kesiapan Investasi

Kelas `RekomendasiInvestasi` menganalisis profil keuangan pengguna berdasarkan dua metrik utama sesuai rumus resmi tim Data Scientist:

- **BLR (Basic Liquidity Ratio)** = Tabungan Total / Total Pengeluaran Bulanan
- **SR (Savings Ratio)** = Tabungan Bulanan / Income Bulanan x 100%

| BLR | Kondisi |
|-----|--------|
| < 1 | Sangat rentan |
| 1 - 3 | Kurang aman |
| 3 - 6 | Cukup sehat |
| > 6 | Sangat siap |

| SR | Kondisi |
|----|--------|
| < 10% | Kurang sehat |
| 10 - 20% | Cukup |
| > 20% | Baik |

| BLR | SR | Status |
|-----|----|--------|
| < 1 | Berapapun | Not Ready |
| 1 - 3 | < 10% | Not Ready |
| 1 - 3 | >= 10% | Moderately Ready |
| > 3 | < 10% | Moderately Ready |
| > 3 | >= 10% | Ready |


In [9]:
class RekomendasiInvestasi:
    """
    Sistem rekomendasi kesiapan investasi FinSmart.

    Menggunakan dua metrik sesuai rumus resmi tim Data Scientist:
    - BLR (Basic Liquidity Ratio) = Tabungan Total / Total Pengeluaran Bulanan
    - SR  (Savings Ratio)         = Tabungan Bulanan / Income Bulanan x 100%

    Input  : profil keuangan pengguna
    Output : status kesiapan investasi dan rekomendasi instrumen
    """

    INSTRUMEN_INVESTASI = {
        "konservatif": [
            "Tabungan berjangka / Deposito",
            "Obligasi Negara Ritel (ORI / SBR)",
            "Reksa Dana Pasar Uang",
        ],
        "moderat": [
            "Reksa Dana Pasar Uang",
            "Reksa Dana Pendapatan Tetap",
            "Obligasi Negara Ritel (ORI / SBR)",
            "P2P Lending terpercaya (OJK)",
        ],
        "agresif": [
            "Reksa Dana Saham",
            "Saham blue-chip (LQ45)",
            "ETF / Index Fund",
            "REITs (properti digital)",
        ],
    }

    def __init__(self, tabungan_total: float, total_pengeluaran_bulanan: float,
                 tabungan_bulanan: float, income_bulanan: float):
        self.tabungan_total            = tabungan_total
        self.total_pengeluaran_bulanan = total_pengeluaran_bulanan
        self.tabungan_bulanan          = tabungan_bulanan
        self.income_bulanan            = income_bulanan

    def hitung_blr(self) -> dict:
        blr = (self.tabungan_total / self.total_pengeluaran_bulanan
               if self.total_pengeluaran_bulanan > 0 else 0)

        if blr < 1:
            kondisi = "Sangat rentan"
        elif blr <= 3:
            kondisi = "Kurang aman"
        elif blr <= 6:
            kondisi = "Cukup sehat"
        else:
            kondisi = "Sangat siap"

        return {"nilai": round(blr, 2), "kondisi": kondisi}

    def hitung_sr(self) -> dict:
        sr = (self.tabungan_bulanan / self.income_bulanan * 100
              if self.income_bulanan > 0 else 0)

        if sr < 10:
            kondisi = "Kurang sehat"
        elif sr <= 20:
            kondisi = "Cukup"
        else:
            kondisi = "Baik"

        return {"nilai_pct": round(sr, 2), "kondisi": kondisi}

    def tentukan_status(self, blr_nilai: float, sr_nilai: float) -> str:
        if blr_nilai < 1:
            return "Not Ready"
        elif blr_nilai <= 3 and sr_nilai < 10:
            return "Not Ready"
        elif blr_nilai <= 3 and sr_nilai >= 10:
            return "Moderately Ready"
        elif blr_nilai > 3 and sr_nilai < 10:
            return "Moderately Ready"
        else:
            return "Ready"

    def tentukan_profil(self, status: str, sr_nilai: float) -> str:
        if status == "Ready":
            return "agresif" if sr_nilai > 20 else "moderat"
        elif status == "Moderately Ready":
            return "moderat"
        else:
            return "konservatif"

    def generate_rekomendasi(self) -> dict:
        blr    = self.hitung_blr()
        sr     = self.hitung_sr()
        status = self.tentukan_status(blr["nilai"], sr["nilai_pct"])
        profil = self.tentukan_profil(status, sr["nilai_pct"])

        pesan = {
            "Not Ready"       : "Fokus dulu membangun dana darurat dan meningkatkan tabungan sebelum investasi.",
            "Moderately Ready": "Kondisi keuangan mulai stabil. Bisa mulai investasi konservatif sambil perkuat tabungan.",
            "Ready"           : "Kondisi keuangan sehat. Siap untuk mulai berinvestasi sesuai profil risiko.",
        }.get(status, "")

        return {
            "ringkasan": {
                "tabungan_total"           : self.tabungan_total,
                "total_pengeluaran_bulanan": self.total_pengeluaran_bulanan,
                "tabungan_bulanan"         : self.tabungan_bulanan,
                "income_bulanan"           : self.income_bulanan,
            },
            "blr"                    : blr,
            "sr"                     : sr,
            "status_kesiapan"        : status,
            "profil_risiko"          : profil,
            "rekomendasi_instrumen"  : self.INSTRUMEN_INVESTASI.get(profil, []),
            "pesan"                  : pesan,
        }

    def tampilkan_laporan(self):
        hasil = self.generate_rekomendasi()

        print("=" * 65)
        print("LAPORAN KESIAPAN INVESTASI — FinSmart")
        print("=" * 65)

        r = hasil["ringkasan"]
        print("\nRINGKASAN KEUANGAN:")
        print(f"  Tabungan Total            : Rp {r['tabungan_total']:>12,.0f}")
        print(f"  Total Pengeluaran Bulanan : Rp {r['total_pengeluaran_bulanan']:>12,.0f}")
        print(f"  Tabungan Bulanan          : Rp {r['tabungan_bulanan']:>12,.0f}")
        print(f"  Income Bulanan            : Rp {r['income_bulanan']:>12,.0f}")

        blr = hasil["blr"]
        sr  = hasil["sr"]
        print("\nMETRIK KEUANGAN:")
        print(f"  BLR (Basic Liquidity Ratio) : {blr['nilai']} — {blr['kondisi']}")
        print(f"  SR  (Savings Ratio)         : {sr['nilai_pct']}% — {sr['kondisi']}")

        print("\nHASIL ANALISIS:")
        print(f"  Status Kesiapan  : {hasil['status_kesiapan']}")
        print(f"  Profil Risiko    : {hasil['profil_risiko'].capitalize()}")
        print(f"  Pesan            : {hasil['pesan']}")

        print("\nREKOMENDASI INSTRUMEN INVESTASI:")
        for instrumen in hasil["rekomendasi_instrumen"]:
            print(f"  - {instrumen}")

        print("=" * 65)
        return hasil


print("Kelas RekomendasiInvestasi berhasil didefinisikan")

Kelas RekomendasiInvestasi berhasil didefinisikan


## 7. Uji Sistem Rekomendasi Kesiapan Investasi

In [10]:
print("=" * 65)
print("UJI KASUS 1: Pengguna A — Karyawan, tabungan cukup")
print("=" * 65)

rek_1 = RekomendasiInvestasi(
    tabungan_total            = 15_000_000,
    total_pengeluaran_bulanan = 3_500_000,
    tabungan_bulanan          = 1_000_000,
    income_bulanan            = 5_000_000,
)
hasil_1 = rek_1.tampilkan_laporan()

UJI KASUS 1: Pengguna A — Karyawan, tabungan cukup
LAPORAN KESIAPAN INVESTASI — FinSmart

RINGKASAN KEUANGAN:
  Tabungan Total            : Rp   15,000,000
  Total Pengeluaran Bulanan : Rp    3,500,000
  Tabungan Bulanan          : Rp    1,000,000
  Income Bulanan            : Rp    5,000,000

METRIK KEUANGAN:
  BLR (Basic Liquidity Ratio) : 4.29 — Cukup sehat
  SR  (Savings Ratio)         : 20.0% — Cukup

HASIL ANALISIS:
  Status Kesiapan  : Ready
  Profil Risiko    : Moderat
  Pesan            : Kondisi keuangan sehat. Siap untuk mulai berinvestasi sesuai profil risiko.

REKOMENDASI INSTRUMEN INVESTASI:
  - Reksa Dana Pasar Uang
  - Reksa Dana Pendapatan Tetap
  - Obligasi Negara Ritel (ORI / SBR)
  - P2P Lending terpercaya (OJK)


In [11]:
print("=" * 65)
print("UJI KASUS 2: Pengguna B — Mahasiswa, tabungan minim")
print("=" * 65)

rek_2 = RekomendasiInvestasi(
    tabungan_total            = 1_500_000,
    total_pengeluaran_bulanan = 1_800_000,
    tabungan_bulanan          = 100_000,
    income_bulanan            = 2_000_000,
)
hasil_2 = rek_2.tampilkan_laporan()

UJI KASUS 2: Pengguna B — Mahasiswa, tabungan minim
LAPORAN KESIAPAN INVESTASI — FinSmart

RINGKASAN KEUANGAN:
  Tabungan Total            : Rp    1,500,000
  Total Pengeluaran Bulanan : Rp    1,800,000
  Tabungan Bulanan          : Rp      100,000
  Income Bulanan            : Rp    2,000,000

METRIK KEUANGAN:
  BLR (Basic Liquidity Ratio) : 0.83 — Sangat rentan
  SR  (Savings Ratio)         : 5.0% — Kurang sehat

HASIL ANALISIS:
  Status Kesiapan  : Not Ready
  Profil Risiko    : Konservatif
  Pesan            : Fokus dulu membangun dana darurat dan meningkatkan tabungan sebelum investasi.

REKOMENDASI INSTRUMEN INVESTASI:
  - Tabungan berjangka / Deposito
  - Obligasi Negara Ritel (ORI / SBR)
  - Reksa Dana Pasar Uang


In [12]:
print("=" * 65)
print("UJI KASUS 3: Pengguna C — Profesional, siap investasi")
print("=" * 65)

rek_3 = RekomendasiInvestasi(
    tabungan_total            = 80_000_000,
    total_pengeluaran_bulanan = 8_000_000,
    tabungan_bulanan          = 4_000_000,
    income_bulanan            = 15_000_000,
)
hasil_3 = rek_3.tampilkan_laporan()

UJI KASUS 3: Pengguna C — Profesional, siap investasi
LAPORAN KESIAPAN INVESTASI — FinSmart

RINGKASAN KEUANGAN:
  Tabungan Total            : Rp   80,000,000
  Total Pengeluaran Bulanan : Rp    8,000,000
  Tabungan Bulanan          : Rp    4,000,000
  Income Bulanan            : Rp   15,000,000

METRIK KEUANGAN:
  BLR (Basic Liquidity Ratio) : 10.0 — Sangat siap
  SR  (Savings Ratio)         : 26.67% — Baik

HASIL ANALISIS:
  Status Kesiapan  : Ready
  Profil Risiko    : Agresif
  Pesan            : Kondisi keuangan sehat. Siap untuk mulai berinvestasi sesuai profil risiko.

REKOMENDASI INSTRUMEN INVESTASI:
  - Reksa Dana Saham
  - Saham blue-chip (LQ45)
  - ETF / Index Fund
  - REITs (properti digital)


In [13]:
with open("contoh_rekomendasi.json", "w", encoding="utf-8") as f:
    json.dump({
        "pengguna_A": hasil_1,
        "pengguna_B": hasil_2,
        "pengguna_C": hasil_3,
    }, f, ensure_ascii=False, indent=2)

print("Contoh rekomendasi disimpan : contoh_rekomendasi.json")

Contoh rekomendasi disimpan : contoh_rekomendasi.json


## 8. Dokumentasi Arsitektur Pipeline AI End-to-End

Seluruh komponen AI didokumentasikan dalam satu file JSON untuk keperluan pelaporan dan referensi tim back-end.


In [14]:
arsitektur = {
    "nama_proyek" : "FinSmart AI Pipeline",
    "versi"       : "2.0.0",
    "ai_engineer" : "Muhammad Syaiful",
    "capstone_id" : "CC26-PSU407",

    "komponen": {

        "1_klasifikasi_pengeluaran": {
            "deskripsi"    : "Model klasifikasi kategori pengeluaran otomatis",
            "model"        : "TensorFlow Functional API + XGBoost (model final)",
            "input"        : "Amount, Payment_Method, Week_Day, Month, Time_Of_Day, MerchantName, Day",
            "output"       : "10 kelas kategori pengeluaran",
            "custom"       : "FinSmartCallback",
            "format_model" : "finsmart_model.keras + xgb_model.pkl",
            "endpoint_api" : "POST /classify",
        },

        "2_behavior_keuangan": {
            "deskripsi"    : "Klasifikasi perilaku keuangan pengguna",
            "model"        : "Decision Tree (financial_type_classifier.pkl) — dari tim Data Scientist",
            "input"        : "Income, Needs, Wants, Savings, Total_Spending, Financial_Balance",
            "output"       : "hemat / normal / boros",
            "acuan"        : "Prinsip 50/30/20",
            "endpoint_api" : "POST /behavior",
        },

        "3_finbot_chatbot": {
            "deskripsi"    : "Chatbot keuangan berbasis NLP",
            "model"        : "Hugging Face — facebook/bart-large-mnli",
            "teknik"       : "Zero-Shot Classification + Intent-Response Mapping",
            "input"        : "Pertanyaan pengguna dalam bahasa Indonesia",
            "output"       : "Respons keuangan berdasarkan intent yang terdeteksi",
            "jumlah_intent": len(INTENT_LABELS),
            "endpoint_api" : "POST /finbot/chat",
        },

        "4_rekomendasi_investasi": {
            "deskripsi"    : "Analisis kesiapan investasi berdasarkan profil keuangan",
            "metode"       : "Rule-Based (BLR + SR)",
            "input"        : "tabungan_total, total_pengeluaran_bulanan, tabungan_bulanan, income_bulanan",
            "output"       : "Status kesiapan investasi + profil risiko + rekomendasi instrumen",
            "acuan"        : "Rumus resmi tim Data Scientist CC26-PSU407",
            "endpoint_api" : "POST /rekomendasi",
        },

        "5_api_serving": {
            "deskripsi" : "REST API untuk menyajikan semua komponen AI ke front-end",
            "framework" : "FastAPI + Uvicorn",
            "versi"     : "2.0.0",
            "endpoints" : [
                "GET  /              — health check",
                "POST /classify      — klasifikasi pengeluaran",
                "POST /behavior      — klasifikasi perilaku keuangan",
                "POST /finbot/chat   — chatbot FinBot",
                "POST /rekomendasi   — kesiapan investasi",
                "GET  /model-info    — info model",
                "GET  /valid-values  — nilai valid tiap fitur",
                "GET  /docs          — Swagger UI",
            ],
            "integrasi" : "Dipanggil oleh back-end Node.js via HTTP",
        },
    },

    "alur_data": [
        "1. Pengguna input transaksi di front-end (React.js)",
        "2. Front-end kirim request ke back-end (Node.js)",
        "3. Back-end forward ke AI Service (FastAPI) via HTTP POST",
        "4. FastAPI load model, lakukan preprocessing, jalankan inference",
        "5. Hasil prediksi dikembalikan ke back-end lalu ke front-end",
        "6. Dashboard menampilkan kategori dan insight kepada pengguna",
    ],

    "file_artifacts": [
        "finsmart_model.keras               — Model TF Functional API",
        "xgb_model.pkl                      — Model XGBoost final (retrain v2)",
        "financial_type_classifier.pkl      — Model Decision Tree behavior (dari DS)",
        "encoders.pkl                       — Label encoders fitur baru (7 fitur)",
        "scaler.pkl                         — StandardScaler",
        "model_metadata.json                — Metadata dan performa model",
        "training_log.json                  — Log training per epoch",
        "finbot_chat_history.json           — Log percakapan FinBot",
        "contoh_rekomendasi.json            — Contoh output rekomendasi investasi",
        "arsitektur_ai.json                 — Dokumentasi arsitektur pipeline",
        "main_complete_v2.py                — FastAPI server lengkap v2",
    ],

    "dataset": {
        "klasifikasi_transaksi": "data_TransactionCategorizationuntukmodel.csv (7 fitur, 8000 baris)",
        "behavior_keuangan"    : "dataset_financial_behavior.csv (6 fitur, 16000 baris)",
        "sumber"               : "Tim Data Scientist CC26-PSU407",
    },
}

with open("arsitektur_ai.json", "w", encoding="utf-8") as f:
    json.dump(arsitektur, f, ensure_ascii=False, indent=2)

print("Dokumentasi arsitektur disimpan : arsitektur_ai.json")
print()
print("ALUR PIPELINE AI END-TO-END:")
for step in arsitektur["alur_data"]:
    print(f"  {step}")
print()
print("KOMPONEN AI (v2):")
for nama, detail in arsitektur["komponen"].items():
    model_info = detail.get("model", detail.get("metode", detail.get("framework", "-")))
    endpoint   = detail.get("endpoint_api", detail.get("endpoints", ["-"])[0])
    print(f"  [{nama}]")
    print(f"    Deskripsi    : {detail['deskripsi']}")
    print(f"    Model/Metode : {model_info}")
    print(f"    Endpoint API : {endpoint}")
    print()

Dokumentasi arsitektur disimpan : arsitektur_ai.json

ALUR PIPELINE AI END-TO-END:
  1. Pengguna input transaksi di front-end (React.js)
  2. Front-end kirim request ke back-end (Node.js)
  3. Back-end forward ke AI Service (FastAPI) via HTTP POST
  4. FastAPI load model, lakukan preprocessing, jalankan inference
  5. Hasil prediksi dikembalikan ke back-end lalu ke front-end
  6. Dashboard menampilkan kategori dan insight kepada pengguna

KOMPONEN AI (v2):
  [1_klasifikasi_pengeluaran]
    Deskripsi    : Model klasifikasi kategori pengeluaran otomatis
    Model/Metode : TensorFlow Functional API + XGBoost (model final)
    Endpoint API : POST /classify

  [2_behavior_keuangan]
    Deskripsi    : Klasifikasi perilaku keuangan pengguna
    Model/Metode : Decision Tree (financial_type_classifier.pkl) — dari tim Data Scientist
    Endpoint API : POST /behavior

  [3_finbot_chatbot]
    Deskripsi    : Chatbot keuangan berbasis NLP
    Model/Metode : Hugging Face — facebook/bart-large-mnli
 

## 9. Ringkasan

In [15]:
print("=" * 65)
print("RINGKASAN — FinSmart FinBot dan Rekomendasi v2")
print("CC26-PSU407 | Muhammad Syaiful")
print("=" * 65)

print("\nKOMPONEN YANG DIKERJAKAN DI NOTEBOOK INI:")
print("  1. FinBot (Zero-Shot Classification)   FinBot class")
print("  2. Rekomendasi Kesiapan Investasi      RekomendasiInvestasi class")
print("     - BLR = Tabungan Total / Pengeluaran Bulanan")
print("     - SR  = Tabungan Bulanan / Income x 100%")
print("  3. Dokumentasi arsitektur pipeline AI  arsitektur_ai.json")

print("\nPERUBAHAN v2 dari v1:")
print("  - Dataset diganti ke dataset baru dari tim DS")
print("  - RekomendasiAnggaran diganti ke RekomendasiInvestasi (rumus BLR + SR)")
print("  - Arsitektur diupdate: tambah komponen /behavior (model DS)")
print("  - Endpoint /rekomendasi selaras dengan main_complete_v2.py")

print("\nFile yang dihasilkan:")
files = [
    ("finbot_chat_history.json",  "Log percakapan FinBot"),
    ("contoh_rekomendasi.json",   "Contoh output kesiapan investasi"),
    ("arsitektur_ai.json",        "Dokumentasi arsitektur pipeline AI"),
]
for fname, desc in files:
    status = "[Ada]" if os.path.exists(fname) else "[Akan dibuat saat run]"
    print(f"  {status} {fname:35s} : {desc}")

print("\n" + "=" * 65)

RINGKASAN — FinSmart FinBot dan Rekomendasi v2
CC26-PSU407 | Muhammad Syaiful

KOMPONEN YANG DIKERJAKAN DI NOTEBOOK INI:
  1. FinBot (Zero-Shot Classification)   FinBot class
  2. Rekomendasi Kesiapan Investasi      RekomendasiInvestasi class
     - BLR = Tabungan Total / Pengeluaran Bulanan
     - SR  = Tabungan Bulanan / Income x 100%
  3. Dokumentasi arsitektur pipeline AI  arsitektur_ai.json

PERUBAHAN v2 dari v1:
  - Dataset diganti ke dataset baru dari tim DS
  - RekomendasiAnggaran diganti ke RekomendasiInvestasi (rumus BLR + SR)
  - Arsitektur diupdate: tambah komponen /behavior (model DS)
  - Endpoint /rekomendasi selaras dengan main_complete_v2.py

File yang dihasilkan:
  [Ada] finbot_chat_history.json            : Log percakapan FinBot
  [Ada] contoh_rekomendasi.json             : Contoh output kesiapan investasi
  [Ada] arsitektur_ai.json                  : Dokumentasi arsitektur pipeline AI

